## 用 GARCH 预测股票未来波动率

- **思路**：用历史日收益率拟合 GARCH(1,1)，得到条件波动率序列，再向前预测未来若干日的波动率。
- **依赖**：`arch`、`pandas`、`numpy`；数据源为 `yfinance`（可改为 OpenBB）。
- **输出**：日度波动率（%）、年化波动率（%），以及历史条件波动率与未来预测的图示。

In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
import logging
import warnings

# 隐藏 matplotlib 字体相关警告（不使用中文时无需 CJK 字体）
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

In [80]:
# GARCH 未来波动率计算
# 使用 GARCH(1,1) 拟合历史收益，并预测未来 1～N 日的波动率（年化/日度）
# 数据获取：优先读 ./data/ 下本地缓存，无则用 OpenBB 下载并落盘
# 多标的、多日期：每个 (symbol, start_date, end_date) 对应一个文件，文件名即“索引”
# 索引方式：get_stock_prices(symbol, start_date, end_date) 对应 ./data/{symbol}_{start}_{end}.csv

from math import sqrt
import os
import glob

DATA_DIR = "./data"

def _cache_path(symbol: str, start_date: str, end_date: str) -> str:
    """本地缓存路径：按 (symbol, start_date, end_date) 索引，end_date=None 记为 latest。"""
    end_str = (end_date if end_date else "latest")
    safe = lambda s: s.replace("-", "") if s else s
    return os.path.join(DATA_DIR, f"{symbol}_{safe(start_date)}_{safe(end_str)}.csv")

def list_cached_data() -> pd.DataFrame:
    """列出 ./data/ 下按 (标的, 起始日期, 结束日期) 索引的价格缓存，便于多标的多日期检索。"""
    os.makedirs(DATA_DIR, exist_ok=True)
    rows = []
    for path in glob.glob(os.path.join(DATA_DIR, "*.csv")):
        name = os.path.basename(path)
        if name.endswith("_iv.csv") or name.endswith("_iv_term.csv") or name.endswith("_iv_series.csv"):
            continue
        base = name[:-4]
        parts = base.split("_")
        if len(parts) != 3:
            continue
        sym, start_raw, end_raw = parts
        start_d = f"{start_raw[:4]}-{start_raw[4:6]}-{start_raw[6:8]}" if len(start_raw) == 8 else start_raw
        end_d = end_raw if end_raw == "latest" else (f"{end_raw[:4]}-{end_raw[4:6]}-{end_raw[6:8]}" if len(end_raw) == 8 else end_raw)
        rows.append({"symbol": sym, "start_date": start_d, "end_date": end_d, "path": path})
    return pd.DataFrame(rows)

def get_stock_prices(symbol: str, start_date: str = "2010-01-01", end_date: str = None,
                     max_retries: int = 3, retry_delay: int = 5, provider="yfinance"):
    """
    按 (symbol, start_date, end_date) 索引：先查 ./data/ 对应缓存，无则下载并落盘；
    同时计算 HV、拉取 IV。返回 (prices, hv_annual_pct, iv_annual_pct)。
    """
    path = _cache_path(symbol, start_date, end_date)
    if os.path.isfile(path):
        s = pd.read_csv(path, index_col=0, parse_dates=True).squeeze("columns")
        close = s.sort_index()
    else:
        import time
        from openbb import obb
        last_err = None
        df = None
        for attempt in range(max_retries):
            try:
                result = obb.equity.price.historical(
                    symbol=symbol,
                    start_date=start_date,
                    end_date=end_date,
                    interval="1d",
                    provider=provider,
                )
                df = result.to_df()
                break
            except Exception as e:
                last_err = e
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                else:
                    raise last_err
        if df is None or df.empty:
            raise ValueError(f"OpenBB 未返回数据: {symbol}")
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(-1)
        close = df["close"] if "close" in df.columns else df["Close"]
        close = close.sort_index()
        os.makedirs(DATA_DIR, exist_ok=True)
        close.to_csv(path)

    ret = returns_from_prices(close)
    # HV：标的过去 30 日历史价格对应的收益率，年化标准差（%）
    ret_30 = ret.tail(30) if len(ret) >= 30 else ret
    hv_annual_pct = float(np.std(ret_30) * np.sqrt(252) * 100)
    iv_annual_pct = get_iv_cached(symbol, provider="yfinance")
    
    return close, hv_annual_pct, iv_annual_pct

def returns_from_prices(prices: pd.Series) -> pd.Series:
    """从价格序列计算对数收益率。"""
    return np.log(prices / prices.shift(1)).dropna()


def _iv_cache_path(symbol: str) -> str:
    """IV 缓存文件：./data/{symbol}_iv.csv，存在则复用，避免重复拉取。"""
    return os.path.join(DATA_DIR, f"{symbol}_iv.csv")


def get_iv_cached(symbol: str, provider="yfinance") -> float:
    """获取标的隐含波动率（年化 %），provider 固定 yfinance。先读本地 ./data/{symbol}_iv.csv，无则拉取并落盘。"""
    path = _iv_cache_path(symbol)
    if os.path.isfile(path):
        try:
            row = pd.read_csv(path)
            iv = float(row["iv_annual_pct"].iloc[0])
            return iv if not np.isnan(iv) else np.nan
        except Exception:
            pass
    try:
        from openbb import obb
        r = obb.derivatives.options.chains(symbol=symbol, provider=provider)
    except Exception:
        return np.nan
    if not r or not getattr(r, "results", None):
        return np.nan
    res = r.results
    iv_list = None
    if hasattr(res, "implied_volatility") and res.implied_volatility is not None:
        iv_list = list(res.implied_volatility)
    else:
        try:
            df = r.to_df()
            if df is not None and not df.empty:
                col = "implied_volatility" if "implied_volatility" in df.columns else "iv"
                if col in df.columns:
                    iv_list = df[col].dropna().tolist()
        except Exception:
            pass
    if not iv_list:
        return np.nan
    iv_list = [x for x in iv_list if x is not None and not (isinstance(x, float) and (np.isnan(x) or x <= 0))]
    if not iv_list:
        return np.nan
    iv_annual_pct = float(np.median(iv_list)) * 100
    os.makedirs(DATA_DIR, exist_ok=True)
    pd.DataFrame({"iv_annual_pct": [iv_annual_pct]}).to_csv(path, index=False)
    return iv_annual_pct


def _iv_term_cache_path(symbol: str) -> str:
    return os.path.join(DATA_DIR, f"{symbol}_iv_term.csv")


def get_iv_term_structure(symbol: str) -> pd.Series:
    """按到期天数 DTE 获取 IV 期限结构（年化 %），用于与未来 horizon 日 GARCH 逐日对比。"""
    path = _iv_term_cache_path(symbol)
    if os.path.isfile(path):
        try:
            df = pd.read_csv(path)
            return df.set_index("dte")["iv_annual_pct"]
        except Exception:
            pass
    try:
        from openbb import obb
        r = obb.derivatives.options.chains(symbol=symbol, provider="yfinance")
    except Exception:
        return pd.Series(dtype=float)
    if not r or not getattr(r, "results", None):
        return pd.Series(dtype=float)
    res = r.results
    try:
        df = r.to_df()
    except Exception:
        return pd.Series(dtype=float)
    if df is None or df.empty:
        return pd.Series(dtype=float)
    iv_col = "implied_volatility" if "implied_volatility" in df.columns else "iv"
    exp_col = None
    for c in ["expiration", "expiry", "expiration_date"]:
        if c in df.columns:
            exp_col = c
            break
    if iv_col not in df.columns or exp_col is None:
        return pd.Series(dtype=float)
    df = df[[exp_col, iv_col]].dropna()
    df["dte"] = (pd.to_datetime(df[exp_col]) - pd.Timestamp.now().normalize()).dt.days
    df = df[df["dte"] >= 0]
    if df.empty:
        return pd.Series(dtype=float)
    by_dte = df.groupby("dte")[iv_col].agg("median") * 100
    os.makedirs(DATA_DIR, exist_ok=True)
    by_dte.to_frame("iv_annual_pct").reset_index().to_csv(path, index=False)
    return by_dte


def _iv_series_cache_path(symbol: str) -> str:
    """IV 时间序列缓存：./data/{symbol}_iv_series.csv，列为 date, iv_annual_pct。"""
    return os.path.join(DATA_DIR, f"{symbol}_iv_series.csv")


def get_iv_series(symbol: str, start_date: str = None, end_date: str = None,
                  update_today: bool = False, provider: str = "yfinance") -> pd.Series:
    """
    获取标的隐含波动率时间序列（年化 %）。
    - 先读本地 ./data/{symbol}_iv_series.csv；若 update_today=True 则拉取当日 IV 并追加写入。
    - 可选：尝试 Intrinio 历史快照（需 API key），见 get_iv_historical_intrinio。
    - start_date/end_date 用于过滤返回的日期范围；None 表示不限制。
    """
    path = _iv_series_cache_path(symbol)
    if update_today:
        iv_today = get_iv_cached(symbol, provider=provider)
        if not np.isnan(iv_today):
            today = pd.Timestamp.now().normalize().strftime("%Y-%m-%d")
            os.makedirs(DATA_DIR, exist_ok=True)
            new_row = pd.DataFrame([{"date": today, "iv_annual_pct": iv_today}])
            if os.path.isfile(path):
                hist = pd.read_csv(path)
                if today not in hist["date"].astype(str).values:
                    hist = pd.concat([hist, new_row], ignore_index=True).drop_duplicates(subset=["date"], keep="last")
                    hist = hist.sort_values("date")
                    hist.to_csv(path, index=False)
            else:
                new_row.to_csv(path, index=False)
    if not os.path.isfile(path):
        return pd.Series(dtype=float)
    try:
        df = pd.read_csv(path)
        df["date"] = pd.to_datetime(df["date"])
        s = df.set_index("date")["iv_annual_pct"].sort_index()
        if start_date is not None:
            s = s[s.index >= pd.Timestamp(start_date)]
        if end_date is not None:
            s = s[s.index <= pd.Timestamp(end_date)]
        return s
    except Exception:
        return pd.Series(dtype=float)


def get_iv_historical_intrinio(symbol: str, start_date: str, end_date: str) -> pd.Series:
    """
    通过 OpenBB Intrinio options snapshots 按日拉取历史 IV（年化 %）。
    需配置 Intrinio API key。若某日无 IV 或接口无 implied_volatility 字段则跳过。
    """
    try:
        from openbb import obb
    except Exception:
        return pd.Series(dtype=float)
    dates = pd.bdate_range(start=start_date, end=end_date or pd.Timestamp.now().strftime("%Y-%m-%d"))
    out = []
    for d in dates:
        try:
            r = obb.derivatives.options.snapshots(date=d.strftime("%Y-%m-%d"), provider="intrinio")
            if not r or not getattr(r, "results", None):
                continue
            df = r.to_df() if hasattr(r, "to_df") else None
            if df is None or df.empty:
                continue
            if "underlying_symbol" in df.columns:
                df = df[df["underlying_symbol"].astype(str).str.upper() == symbol.upper()]
            if df.empty:
                continue
            iv_col = "implied_volatility" if "implied_volatility" in df.columns else "iv"
            if iv_col not in df.columns:
                continue
            vals = df[iv_col].dropna().replace([np.inf, -np.inf], np.nan).dropna()
            if vals.empty:
                continue
            iv_pct = float(vals.median()) * 100
            if iv_pct <= 0:
                continue
            out.append({"date": d, "iv_annual_pct": iv_pct})
        except Exception:
            continue
    if not out:
        return pd.Series(dtype=float)
    df = pd.DataFrame(out).set_index("date")["iv_annual_pct"].sort_index()
    path = _iv_series_cache_path(symbol)
    os.makedirs(DATA_DIR, exist_ok=True)
    df.to_frame().reset_index().to_csv(path, index=False)
    return df


def get_volatility_metrics(symbol: str, start_date: str = "2010-01-01", end_date: str = None,
                          horizon_days: int = 22, hold_period: int = 1):
    """
    一站式：价格+HV+IV（get_stock_prices 统一 yfinance 并缓存）→ GARCH 拟合与预测。
    返回 dict：prices, returns, fit, forecast, hv_annual_pct, iv_annual_pct,
    garch_1d_pct, garch_nd_pct, vol_table, cond_vol_pct, annual_vol_pct,
    cond_vol_hist, garch_vol_series, hv_series, horizon_days。
    """
    prices, hv_annual_pct, iv_annual_pct = get_stock_prices(symbol, start_date, end_date)
    returns = returns_from_prices(prices)
    # 滚动 30 日历史波动率时间序列（年化 %），与 IV 时间序列对齐对比
    roll_window = horizon_days
    hv_series = returns.rolling(roll_window).std() * np.sqrt(252) * 100
    hv_series = hv_series.dropna()

    # 计算IV日度
    iv_daily_pct = (iv_annual_pct / 100) / np.sqrt(252)
    # 用最后一日的price计算预期波动
    last_price = prices.iloc[-1] if hasattr(prices, "iloc") else prices[-1]

    # GARCH 预测
    model = arch_model(returns * 100, mean="Constant", vol="GARCH", p=1, q=1, rescale=False)
    fit = model.fit(disp="off")
    forecast = fit.forecast(horizon=horizon_days, reindex=False)
    cond_var = forecast.variance.values[-1, :]
    cond_vol_pct = np.sqrt(cond_var)
    annual_vol_pct = cond_vol_pct * np.sqrt(252)
    garch_1d_pct = float(annual_vol_pct[0])
    garch_nd_pct = float(annual_vol_pct[-1])

    # 用计算预期波动
    # expected_move = iv_daily_pct * np.sqrt(horizon_days * hold_period) * last_price
    expected_move = (garch_nd_pct / np.sqrt(252)) * 0.01 * np.sqrt(horizon_days * hold_period) * last_price
    lower_bound = round(last_price - expected_move, 2)
    upper_bound = round(last_price + expected_move, 2)

    vol_table = pd.DataFrame({
        "HV annual %": [round(hv_annual_pct, 2)],
        "IV annual %": [round(iv_annual_pct, 2) if not np.isnan(iv_annual_pct) else "—"],
        "GARCH T+1 annual %": [round(garch_1d_pct, 2)],
        f"GARCH T+{horizon_days} annual %": [round(garch_nd_pct, 2)],
        "price": [round(last_price, 2)],
        "expected move": [round(expected_move, 2)],
        "expected range": [f"{lower_bound} ~ {upper_bound}"]
    })
    vol_table.index = [symbol]

    # GARCH 条件波动率时间序列（年化 %），与 returns 同索引，用于与 HV 同图对比
    garch_vol_series = fit.conditional_volatility * np.sqrt(252)

    return {
        "prices": prices,
        "returns": returns,
        "fit": fit,
        "forecast": forecast,
        "hv_annual_pct": hv_annual_pct,
        "iv_annual_pct": iv_annual_pct,
        "garch_1d_pct": garch_1d_pct,
        "garch_nd_pct": garch_nd_pct,
        "vol_table": vol_table,
        "cond_vol_pct": cond_vol_pct,
        "annual_vol_pct": annual_vol_pct,
        "cond_vol_hist": fit.conditional_volatility,
        "garch_vol_series": garch_vol_series,
        "hv_series": hv_series,
        "horizon_days": horizon_days,
    }

In [76]:
def plot_hv_garch(result: dict, symbol: str, figsize=(10, 5)):
    """绘制 HV 历史序列、GARCH 条件波动率历史及未来预测。"""
    last_dt = result["returns"].index[-1]
    h = result["horizon_days"]
    future_dates = [last_dt + pd.Timedelta(days=i) for i in range(1, h + 1)]
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(result["hv_series"].index, result["hv_series"].values, label="HV, 30 rolling days", color="C0", lw=1.2)
    ax.plot(result["garch_vol_series"].index, result["garch_vol_series"].values, label="GARCH cond_vol_hist", color="C1", linestyle="--", lw=1.2)
    ax.plot(future_dates, result["annual_vol_pct"], label="GARCH pred_vol", color="C2", marker="o", ms=3, lw=1.2)
    ax.set_ylabel("annual_vol_pct")
    ax.set_xlabel("date")
    ax.set_title(f"{symbol}: HV and GARCH hist & pred")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [65]:
# 配置：标的与样本区间；一次调用完成价格(缓存)、HV、IV(缓存)、GARCH
SYMBOL = "AAPL"
START = "2024-01-01"
END = None
HORIZON_DAYS = 25

result = get_volatility_metrics(SYMBOL, START, END, horizon_days=HORIZON_DAYS)
prices = result["prices"]
returns = result["returns"]
fit = result["fit"]
forecast = result["forecast"]
annual_vol_pct = result["annual_vol_pct"]
cond_vol_pct = result["cond_vol_pct"]
vol_table = result["vol_table"]
cond_vol_hist = result["cond_vol_hist"]
garch_vol_series = result["garch_vol_series"]
hv_series = result["hv_series"]

In [68]:
# GARCH 拟合结果（已在 get_volatility_metrics 内完成）
print(fit.summary())

                     Constant Mean - GARCH Model Results                      
Dep. Variable:                  close   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -1022.00
Distribution:                  Normal   AIC:                           2052.01
Method:            Maximum Likelihood   BIC:                           2069.17
                                        No. Observations:                  540
Date:                Sat, Feb 28 2026   Df Residuals:                      539
Time:                        15:45:50   Df Model:                            1
                                 Mean Model                                
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
mu             0.0648  6.768e-02      0.957      0.338 [-6.78

### 比较与验证模型
- 评估预测准确性：将GARCH预测值与未来的已实现波动率（未来HV）进行比较，检验模型的预测能力。也可以对比IV与未来实现波动率，判断期权定价是否合理。

- 识别系统性偏差：如果IV长期高于GARCH预测或未来实现波动率，可能说明市场存在恐慌溢价；反之则可能低估风险。

- 观察波动率期限结构：比较不同期限的HV、IV和GARCH预测，可以了解短期与长期波动预期的差异，判断市场对近期事件的反应。

### 波动率套利（Volatility Arbitrage）：

- 当IV显著高于GARCH预测的未来波动率 时，意味着期权可能被高估，可考虑卖出期权（做空波动率），同时用标的资产对冲Delta风险，等待IV回归。

- 当IV显著低于GARCH预测 时，期权可能被低估，可考虑买入期权（做多波动率）。
注意：需要设置合理的阈值和仓位管理，因为GARCH预测也可能有误差。

### 方向性交易辅助：

- 如果GARCH预测波动率即将上升，而当前市场情绪平稳（IV较低），可提前布局波动率多头（如跨式期权）。

- 如果HV已处于历史高位，且GARCH预测将回落，同时IV仍高企，可考虑做空波动率。

In [81]:
start_date = "2025-01-01"
horizon_days = 30

for sym in ["AAPL", "MSFT", "SPY", "QQQ", "DIA", "KO"]:
    result = get_volatility_metrics(sym, start_date, end_date=None, horizon_days=horizon_days, hold_period=0.66)
    print(f"标的 {sym} 波动率对比（HV / IV / GARCH）")
    display(result["vol_table"])
    # plot_hv_garch(result, sym)

标的 AAPL 波动率对比（HV / IV / GARCH）


,HV annual %,IV annual %,GARCH T+1 annual %,GARCH T+30 annual %,price,expected move,expected range
AAPL,30.79,37.24,32.26,32.08,264.18,23.76,240.42 ~ 287.94


标的 MSFT 波动率对比（HV / IV / GARCH）


,HV annual %,IV annual %,GARCH T+1 annual %,GARCH T+30 annual %,price,expected move,expected range
MSFT,42.25,34.17,33.18,36.88,392.74,40.61,352.13 ~ 433.35


标的 SPY 波动率对比（HV / IV / GARCH）


...

,HV annual %,IV annual %,GARCH T+1 annual %,GARCH T+30 annual %,price,expected move,expected range
QQQ,17.23,25.35,17.8,21.23,607.29,36.13,571.16 ~ 643.42


标的 DIA 波动率对比（HV / IV / GARCH）


,HV annual %,IV annual %,GARCH T+1 annual %,GARCH T+30 annual %,price,expected move,expected range
DIA,13.77,20.9,14.15,15.15,489.66,20.8,468.86 ~ 510.46


标的 KO 波动率对比（HV / IV / GARCH）


,HV annual %,IV annual %,GARCH T+1 annual %,GARCH T+30 annual %,price,expected move,expected range
KO,16.13,28.44,15.67,15.54,81.56,3.55,78.01 ~ 85.11
